<a href="https://colab.research.google.com/github/YuenZJ/Kitmodel/blob/main/BAGEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!sudo apt-get update -y
!sudo apt-get install python3.12

!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 2


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,813 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (

In [9]:
# install pip for new python
!sudo apt-get install python3.12-distutils
!wget https://bootstrap.pypa.io/get-pip.py
!python get-pip.py

# credit of these last two commands blongs to @Erik
# install colab's dependencies
!python -m pip install ipython ipython_genutils ipykernel jupyter_console prompt_toolkit httplib2 astor

# link to the old google package
!ln -s /usr/local/lib/python3.11/dist-packages/google \
       /usr/local/lib/python3.12/dist-packages/google

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package python3.12-distutils
E: Couldn't find any package by glob 'python3.12-distutils'
E: Couldn't find any package by regex 'python3.12-distutils'
--2026-04-14 07:51:21--  https://bootstrap.pypa.io/get-pip.py
Resolving bootstrap.pypa.io (bootstrap.pypa.io)... 151.101.0.175, 151.101.64.175, 151.101.128.175, ...
Connecting to bootstrap.pypa.io (bootstrap.pypa.io)|151.101.0.175|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2193439 (2.1M) [text/x-python]
Saving to: ‘get-pip.py.2’

get-pip.py.2        100%[===================>]   2.09M  --.-KB/s    in 0.03s   

2026-04-14 07:51:21 (60.4 MB/s) - ‘get-pip.py.2’ saved [2193439/2193439]

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
   

In [15]:
!pip install "biobagel[local] @ git+https://github.com/softnanolab/bagel.git" "transformers>=4.49.0,<5.0.0"
!pip uninstall -y torchvision

  Cloning https://github.com/softnanolab/bagel.git to /tmp/pip-install-m__j7ig6/biobagel_f09dceef420f4d17861646cc18b8efb4
  Running command git clone --filter=blob:none --quiet https://github.com/softnanolab/bagel.git /tmp/pip-install-m__j7ig6/biobagel_f09dceef420f4d17861646cc18b8efb4
  Resolved https://github.com/softnanolab/bagel.git to commit 4bf79b6c7c1d8196a0b6b71412b3be674fc3ec74
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [19]:
# Monkey-patch: inject imports that Modal's esm_image.imports() doesn't provide locally
from transformers import AutoTokenizer, EsmForProteinFolding
from transformers.models.esm.modeling_esmfold import EsmFoldingTrunk
import torch
import boileroom.models.esm.esmfold as _esmfold_mod
_esmfold_mod.AutoTokenizer = AutoTokenizer
_esmfold_mod.EsmForProteinFolding = EsmForProteinFolding
_esmfold_mod.EsmFoldingTrunk = EsmFoldingTrunk
_esmfold_mod.torch = torch
print("Patched esmfold module successfully")

# To resolve the DeprecationError, you generally need to update the packages.
# You can try re-installing biobagel which should bring in the latest compatible boileroom and modal versions.
# If you run this cell, make sure to run the installation cell (x1CBifOPoK1X) again as well.
# !pip install --upgrade "biobagel[local] @ git+https://github.com/softnanolab/bagel.git" "transformers>=4.49.0,<5.0.0"

DeprecationError: Deprecated on 2025-02-24: The 'container_idle_timeout' parameter has been renamed to 'scaledown_window'.

In [18]:
import random
import bagel as bg
import os
from typing import Any

# Get the value of an environment variable
use_modal = False

# Check
print(f'Whether to use modal: {use_modal}')

# Define the target protein
# This sequence comes from a PDB of the interleukin-8 protein
# >1IL8_1|Chains A, B|INTERLEUKIN-8|Homo sapiens (9606)
target_sequence = 'SAKELRCQCIKTYSKPFHPKFIKELRVIESGPHCANTEIIVKLSDGRELCLDPKENWVQRVVEKFLKRAENS'

# Define the mutability of the residues, all immutable in this case since this is the target sequence
mutability = [False for _ in range(len(target_sequence))]

# Define a chain providing a list of residues
residues_target = [
    bg.Residue(name=aa, chain_ID='Maxi', index=i, mutable=mut)
    for i, (aa, mut) in enumerate(zip(target_sequence, mutability))
]

# Define residues in the hotspot where you want to bind. Here we choose those between residues 10-20
residues_hotspot = [residues_target[i] for i in range(10, 20)]
target_chain = bg.Chain(residues=residues_target)

# For the binder, start with a random sequence of 10 residues
binder_length = 10
binder_sequence = ''.join([random.choice(list(bg.constants.aa_dict.keys())) for _ in range(binder_length)])

# Define the mutability of the residues, all mutable in this case since this is the design sequence
mutability = [True for _ in range(len(target_sequence))]

# Define the chain
residues_binder = [
    bg.Residue(name=aa, chain_ID='Stef', index=i, mutable=mut)
    for i, (aa, mut) in enumerate(zip(binder_sequence, mutability))
]
binder_chain = bg.Chain(residues=residues_binder)

# Define the FoldingOracle
# See https://openreview.net/forum?id=g8S53BmXE6 for linker parameter tuning
config = {
    'glycine_linker': 'GGGG',
    'position_ids_skip': 1024,
}
esmfold = bg.oracles.ESMFold(use_modal=use_modal, config=config)

# Define the energy terms to be applied to the chain. apply them to residues, and specify the weight
energy_terms = [
    bg.energies.PTMEnergy(
        oracle=esmfold,
        weight=1.0,
    ),
    bg.energies.OverallPLDDTEnergy(
        oracle=esmfold,
        weight=1.0,
    ),
    bg.energies.HydrophobicEnergy(
        oracle=esmfold,
        weight=5.0,
    ),
    bg.energies.PAEEnergy(
        oracle=esmfold,
        residues=[residues_hotspot, residues_binder],
        weight=5.0,
    ),
    bg.energies.SeparationEnergy(
        oracle=esmfold,
        residues=[residues_hotspot, residues_binder],
        weight=1.0,
    ),
]

# Define the state
state = bg.State(
    name='state_A',
    chains=[binder_chain, target_chain],
    energy_terms=energy_terms,
)

# Define the system
initial_system = bg.System(states=[state])

# Define the minimizer
# Simulated tempering does n_steps_low at a low temperature (enhancing local minimization),
# and n_steps_high at a high temperature (exploring the space)
minimizer = bg.minimizer.SimulatedTempering(
    mutator=bg.mutation.Canonical(n_mutations=1), # cannot add/remove residues, only substitutes for different amino acid types
    high_temperature=2,
    low_temperature=0.1,
    n_cycles=10,
    n_steps_low=100,
    n_steps_high=20,
    log_frequency=50,
)

best_system = minimizer.minimize_system(system=initial_system)


Whether to use modal: False


DeprecationError: Deprecated on 2025-02-24: The 'container_idle_timeout' parameter has been renamed to 'scaledown_window'.